# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, which includes full field and schema semantics for advanced, machine-readable data processing.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load and inspect dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\nDescription: {metadata.description}\nVersion: {metadata.version}")

## 2. Data Overview
Let us review available record sets, their IDs (`@id`), and the fields (columns) they contain.

In [ ]:
# List all record sets and their fields, referencing each by their `@id`.
print("Available record sets and fields:")
for rs in dataset.metadata.record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
We will load the available record set(s) into a pandas DataFrame for further analysis, always referencing entities by their `@id`.

In [ ]:
# Extract data from each record set by `@id`
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for Record Set {record_set_id} with shape {dataframes[record_set_id].shape}")

# For demonstration, select the first record set
if len(record_sets_ids) > 0:
    main_record_set_id = record_sets_ids[0]
    print("Available columns:", dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Let us filter on a numeric field, normalize it, and group by a categorical field. All fields referenced use their `@id`.

> **Note:** Field IDs can be found in the overview above. Replace with appropriate values as needed.

In [ ]:
# Example EDA, using actual field `@id`s (update if needed)
if len(record_sets_ids) > 0:
    df = dataframes[main_record_set_id]

    # Attempt to auto-select a numeric field and a grouping field by data type
    numeric_field_id = None
    group_field_id = None

    for field in dataset.metadata.record_sets[0].fields:
        # Looking for Float/Integer types
        if field.data_type in ('Number', 'Float', 'Integer') and numeric_field_id is None:
            numeric_field_id = field.id
        # Looking for Text/categorical
        if field.data_type == 'Text' and group_field_id is None:
            group_field_id = field.id

    if numeric_field_id is not None and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalizing the numeric field
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        if group_field_id is not None and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found by @id for EDA in this record set.")
else:
    print("No data available for EDA.")

## 5. Visualization

Let us plot the distribution of the selected numeric variable and its normalized values. (Requires matplotlib.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets_ids) > 0 and numeric_field_id is not None and numeric_field_id in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(df[numeric_field_id], bins=30, kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field_id}")

    if numeric_field_id + '_normalized' in filtered_df.columns:
        sns.histplot(filtered_df[numeric_field_id + '_normalized'], bins=30, kde=True, ax=axes[1], color='orange')
        axes[1].set_title(f"Normalized {numeric_field_id} (Filtered)")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- We loaded the FAIR² human mast cell mass spectrometry dataset using its Croissant schema and `mlcroissant`.
- We inspected the dataset metadata and dynamically explored all available record sets and fields by their `@id`.
- Key numeric variables were explored and normalized, with optional grouping and visualization.

This workflow can be adapted for any Croissant-formatted dataset for robust, FAIR data analysis.